# Experiments 2–3 Full-Budget 16K & DEV 32K Evaluation (Colab A100 80GB / 40GB)

Strictly follows `eval_multi_turn/EXPERIMENT_CONTRACT.md`:
- **3 Open-Weight Reasoning Models (BF16, pinned HF commits)**: `Qwen/Qwen3-8B`, `Qwen/Qwen3-14B`, `zai-org/GLM-Z1-9B-0414`
- **5 Experiment-2 Protocols**: `FULL`, `CONCAT`, `CONCAT_MATCHED`, `DYN_FREE`, `DYN_REQUEST` *(Exp-4B `DYN_REQUEST_GUARD` deferred)*
- **Calibration Cells on `DEV 20`**: `16K` (`eval_multi_turn/runs/runs_16k/`) + `32K` (`eval_multi_turn/runs/runs_32k_dev/`) before `TEST 100`
- **Isolated Output Directories**: preserves prior `8K` dataset in `eval_multi_turn/runs/` untouched

In [ ]:
# 1. Mount Drive, Unzip Code Only (runs/ excluded), chdir, & Check GPU
import os
import torch
from google.colab import drive

drive.mount('/content/drive')
!unzip -q -o /content/drive/MyDrive/cscwd_v2_colab.zip -d /content/drive/MyDrive/
WORK_DIR = '/content/drive/MyDrive/cscwd_v2'
os.chdir(WORK_DIR)
print('Working dir:', os.getcwd())
!nvidia-smi
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU VRAM: {vram:.2f} GiB')

In [ ]:
# 2. Install runtime dependencies & verify all 5 offline unit/mock tests
import os
os.chdir('/content/drive/MyDrive/cscwd_v2')
!pip -q install -U vllm sympy huggingface_hub openai
print('Confirmed CWD:', os.getcwd())
!python eval_multi_turn/protocol_utils.py --self-test
!python eval_multi_turn/rollout_vllm.py --mock-test
!python eval_multi_turn/sympy_answer_grader.py --self-test
!python eval_multi_turn/judge_trajectories.py --mock-test
!python eval_multi_turn/analyze_experiments.py --mock-test

In [ ]:
# 3. [DEV 20 16K CALIBRATION] Run all 3 models (reps=1) cleanly into runs/runs_16k/
import glob
import json
import os

os.chdir('/content/drive/MyDrive/cscwd_v2')
MODELS = [
    'Qwen/Qwen3-8B',
    'zai-org/GLM-Z1-9B-0414',
    'Qwen/Qwen3-14B',
]
for m in MODELS:
    tag = m.split('/')[-1].lower().replace('-', '_')
    !python eval_multi_turn/rollout_vllm.py \
      --split DEV \
      --plan eval_multi_turn/disclosure_plan_dev20.jsonl \
      --model {m} \
      --protocols FULL CONCAT CONCAT_MATCHED DYN_FREE DYN_REQUEST \
      --reps 1 \
      --max-tokens 16384 \
      --max-model-len 32768 \
      --output eval_multi_turn/runs/runs_16k/dev20_{tag}_16384.jsonl \
      --resume

print('\n=== DEV 20 16K SUMMARY ===')
for p in sorted(glob.glob('eval_multi_turn/runs/runs_16k/dev20_*_16384_meta.json')):
    meta = json.load(open(p, encoding='utf-8'))
    trunc = sum(v['empty_truncated_turns'] for v in meta['by_protocol'].values())
    clamp = sum(v['budget_clamped_turns'] for v in meta['by_protocol'].values())
    print(os.path.basename(p), '| completed:', meta['completed_trajectories'], '| empty_truncated_turns:', trunc, '| budget_clamped_turns:', clamp)

In [ ]:
# 4. [DEV 20 32K VALIDATION] Run all 3 models (reps=1) with max_tokens=32768 into runs/runs_32k_dev/ & compare 8K / 16K / 32K
import glob
import json
import os

os.chdir('/content/drive/MyDrive/cscwd_v2')
MODELS_32K = [
    ('Qwen/Qwen3-8B', 40960),
    ('zai-org/GLM-Z1-9B-0414', 32768),
    ('Qwen/Qwen3-14B', 40960),
]
for m, ctx_len in MODELS_32K:
    tag = m.split('/')[-1].lower().replace('-', '_')
    !python eval_multi_turn/rollout_vllm.py \
      --split DEV \
      --plan eval_multi_turn/disclosure_plan_dev20.jsonl \
      --model {m} \
      --protocols FULL CONCAT CONCAT_MATCHED DYN_FREE DYN_REQUEST \
      --reps 1 \
      --max-tokens 32768 \
      --max-model-len {ctx_len} \
      --output eval_multi_turn/runs/runs_32k_dev/dev20_{tag}_32768.jsonl \
      --resume

print('\n=== DEV 20 MULTI-BUDGET COMPARISON (8K vs 16K vs 32K) ===')
meta_paths = (
    sorted(glob.glob('eval_multi_turn/runs/dev20/*_meta.json'))
    + sorted(glob.glob('eval_multi_turn/runs/runs_16k/dev20_*_meta.json'))
    + sorted(glob.glob('eval_multi_turn/runs/runs_32k_dev/dev20_*_meta.json'))
)
for p in meta_paths:
    meta = json.load(open(p, encoding='utf-8'))
    dcfg = meta['run_signature']['decoding_cfg']
    by_p = meta['by_protocol']
    trunc = sum(v['empty_truncated_turns'] for v in by_p.values())
    clamp = sum(v['budget_clamped_turns'] for v in by_p.values())
    per_p = {k: by_p[k]['empty_truncated_turns'] for k in sorted(by_p)}
    print(
        f"{os.path.basename(p):36s} | max_tok={dcfg['max_tokens']:5d} | ctx={dcfg['max_model_len']:5d} "
        f"| done={meta['completed_trajectories']:3d} | trunc={trunc:2d} {per_p} | clamp={clamp} | {meta['elapsed_sec']}s"
    )

In [ ]:
# 5. [MAIN TEST 100 16K ROLLOUT] Run all 3 models (reps=3) into runs/runs_16k/
# If DEV 32K shows major gain & you prefer 32768 on TEST too, switch --max-tokens 32768, --max-model-len (40960 for Qwen3 / 32768 for GLM), and output folder runs/runs_32k/.
import os
os.chdir('/content/drive/MyDrive/cscwd_v2')
MODELS = [
    'Qwen/Qwen3-8B',
    'zai-org/GLM-Z1-9B-0414',
    'Qwen/Qwen3-14B',
]
for m in MODELS:
    tag = m.split('/')[-1].lower().replace('-', '_')
    !python eval_multi_turn/rollout_vllm.py \
      --split TEST \
      --plan eval_multi_turn/disclosure_plan_test100.jsonl \
      --model {m} \
      --protocols FULL CONCAT CONCAT_MATCHED DYN_FREE DYN_REQUEST \
      --reps 3 \
      --max-tokens 16384 \
      --max-model-len 32768 \
      --output eval_multi_turn/runs/runs_16k/test100_{tag}_16384.jsonl \
      --resume

In [ ]:
# 6. Auto-release A100 runtime immediately after Cell 5 finishes to save compute units
from google.colab import runtime
runtime.unassign()

In [ ]:
# 7. Local Judge & Statistical Aggregation Reference (run locally after downloading runs_16k/)
# !python eval_multi_turn/judge_trajectories.py --split TEST --rollouts eval_multi_turn/runs/runs_16k/test100_qwen3_14b_16384.jsonl --output eval_multi_turn/runs/runs_16k/judged_qwen3_14b_16384.jsonl
# !python eval_multi_turn/analyze_experiments.py --inputs eval_multi_turn/runs/runs_16k/judged_*_16384.jsonl --output eval_multi_turn/runs/runs_16k/exp2_4_summary_16384.json